# B1：品質遮罩整合驗收（唯讀）

直接按「執行階段 → 全部執行」，約 3–8 分鐘。過程中會跳出 Drive 授權視窗，請按允許。

**檢查項目**（模組 `tw_lake.audit.integration_check`）

1. 全部輸出檔與輸入檔的 SHA-256 是否與 `quality_manifest.json` 一致。
2. 遮罩的 code commit 是否為 `fbec159`；富邦 5 分K 目前 published 的版本序號是否仍為 68。
3. 重新計算四個層級的股票日數，必須與 manifest 一致：
   - 全部股票日 1,788,899
   - 價格研究 1,733,539
   - 量能研究 1,707,922
   - Level A 212,428
4. 12 條規則檢查，例如：UNKNOWN 不能算可用、隔離日期不能出現在可用樣本、量能樣本必須是價格樣本的子集。
5. 抽 300 個可用股票日，與 5 分K 原始分區逐根比對 K 棒數與成交量。

**寫入範圍**

- 不修改任何資料、帳本、manifest 或遮罩。
- 只在 Drive 的 `audits/integration_check/` 下建立一個新資料夾，存放驗收結果。

最後會印出 **PASS** 或 **FAIL**，以及失敗原因。

In [ ]:
# 1. 參數
REPO_OWNER, REPO_NAME, BRANCH = "hh4832", "tw-market-data-lake", "main"
DATA_ROOT = "/content/drive/MyDrive/00Quant_Research/data"
QDIR = f"{DATA_ROOT}/_tw-market-data-lake/audits/quality_acceptance/quality_mask_20261030_014633_fbec159"
EXPECTED_COMMIT = "fbec159"     # 產生遮罩的 code commit
EXPECTED_SEQ = 68               # 遮罩所依據的 fubon_5m published seq
SAMPLE = 300                    # K 棒抽查的股票日數
OUT_DIR = f"{DATA_ROOT}/_tw-market-data-lake/audits/integration_check"
PROJECT_DIR = f"/content/{REPO_NAME}"

In [ ]:
# 2. 掛載 Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3. clone / pull + 依賴 + 依賴
import subprocess, os
if os.path.exists(f"{PROJECT_DIR}/.git"):
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git", PROJECT_DIR], check=True)
print("HEAD:", subprocess.check_output(["git", "-C", PROJECT_DIR, "rev-parse", "HEAD"], text=True).strip())
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml>=6", "tqdm>=4.66"], check=True)
def stream(args):
    env = dict(os.environ, PYTHONPATH=f"{PROJECT_DIR}/src", PYTHONUNBUFFERED="1")
    p = subprocess.Popen(args, cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

def ensure_drive():
    """Mount Drive safely. If an earlier cell unmounted Drive and then wrote under /content/drive,
    those files are only on this runtime's disk: move them aside (never delete) and copy any audit
    folders that are missing on the real Drive."""
    import os, shutil, time
    from google.colab import drive
    if os.path.ismount("/content/drive"):
        return
    stray = None
    if os.path.isdir("/content/drive") and os.listdir("/content/drive"):
        stray = f"/content/drive_local_{int(time.time())}"
        shutil.move("/content/drive", stray)
        print("moved local (unsynced) files to", stray)
    drive.mount("/content/drive")
    if stray:
        src_root = f"{stray}/MyDrive/00Quant_Research/data/_tw-market-data-lake/audits"
        dst_root = f"{DATA_ROOT}/_tw-market-data-lake/audits"
        for dirpath, dirnames, filenames in os.walk(src_root):
            rel = os.path.relpath(dirpath, src_root)
            for d in list(dirnames):
                target = os.path.join(dst_root, rel, d)
                if not os.path.exists(target):
                    shutil.copytree(os.path.join(dirpath, d), target)
                    print("copied missing", target)
                    dirnames.remove(d)
print("Python", sys.version.split()[0])

In [ ]:
# 4. 執行驗收（唯讀）
assert os.path.exists(f"{QDIR}/quality_manifest.json"), f"找不到 {QDIR}"
rc = stream([sys.executable, "-m", "tw_lake.audit.integration_check", "--qdir", QDIR, "--data-root", DATA_ROOT,
             "--expected-commit", EXPECTED_COMMIT, "--expected-seq", str(EXPECTED_SEQ), "--sample", str(SAMPLE),
             "--out-dir", OUT_DIR])
print("exit", rc, "(0 = PASS)")

In [ ]:
# 5. 確保結果上傳到 Drive
from google.colab import drive
drive.flush_and_unmount()
print("結果已存到", OUT_DIR)